<a href="https://colab.research.google.com/github/kanchanraiii/SecureRag/blob/master/BuildKB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu sentence-transformers presidio-analyzer presidio-anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 10.8 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
import glob

# Initialize Presidio and embeddings model
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def redact_text(text):
    results = analyzer.analyze(text=text, language='en')
    return anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={"DEFAULT": OperatorConfig("replace", {"new_value": "[REDACTED]"})}
    ).text

raw_docs = []
for file in glob.glob("datasets/*.jsonl"):
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            record_text = " ".join([f"{k}: {v}" for k, v in obj.items()])
            raw_docs.append(record_text)

print(f"Loaded {len(raw_docs)} total documents.")

# Redact all text
redacted_docs = [redact_text(doc) for doc in raw_docs]

# Embed
embeddings = embed_model.encode(redacted_docs, show_progress_bar=True).astype("float32")

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, "secure_knowledge_base.index")

# Save the redacted text
with open("redacted_docs.json", "w") as f:
    json.dump(redacted_docs, f)

print("✅ Built secure knowledge base successfully.")


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 36000 total documents.
